## ECG Spectral Analysis ##

This is a comprehensive reference guide for Spectral ECG Analysis, synthesizing the definitions, Python implementations, and heuristic behaviors for identifying **NSR, AFIB, AFL, VFL, and VF**.

We will use a unified Python class structure. This code assumes you have already performed the FFT and calculated the Power Spectral Density (PSD) for the **0.5 – 20 Hz** range (or similar).

#### The Metrics

| Metric | Definition | Interpretation |
| --- | --- | --- |
| **Spectral Centroid** | The center of mass of the power spectrum. | Shifts higher during Tachycardia/VFL/VF. |
| **Spectral Spread** | The standard deviation of the power spectrum around the centroid. | Higher spread indicates more chaotic rhythms (VF). |
| **Spectral Entropy** | Shannon entropy of the normalized PSD (treated as a probability distribution). | **0:** Pure sine wave. **1:** White noise (VF). |
| **Spectral Flatness** | Ratio of Geometric Mean to Arithmetic Mean of the PSD. | **0:** Spiky (Organized). **1:** Flat (Disorganized/VF). |
| **Spectral Edge ($f_{95}$)** | Frequency below which 95% of total power is contained. | Indicates the "bandwidth" of the signal. |
| **Dominant Freq ($f_{dom}$)** | The frequency bin with the highest power. | The primary heart rate (or flutter rate). |
| **Spectral Peaks** | Local maxima in the PSD. | Candidate frequencies for fundamental and harmonics. |
| **Base & Harmonics** | $f_{dom}$ is the "Base." Peaks near $2 \times f_{dom}, 3 \times f_{dom}$ are "Harmonics." | Presence of harmonics indicates an organized, non-sinusoidal rhythm (like QRS). |
| **HNR** | Ratio (in dB) of energy in Harmonic peaks vs. background noise. | High = Organized. Low = Chaotic/VF. |
| **VF Filter Leak** | Ratio of energy remaining after strictly removing the dominant peak and harmonics. | High = Disorganized energy (VF). Low = Clean signal. |
| **Band Power Ratio** | Ratio of Power in High Band ($5\text{--}20\text{ Hz}$) to Low Band ($0.5\text{--}5\text{ Hz}$). | VF has more energy in the high band. |


---

### Rhythm Behavior Matrix

How these metrics move across different cardiac states.

| Metric | **NSR** (Normal) | **AFIB** (Atrial Fib) | **AFL** (Atrial Flutter) | **VFL** (Vent. Flutter) | **VF** (Vent. Fib) |
| --- | --- | --- | --- | --- | --- |
| **Centroid** | Low | Low-Mid | Mid | **Mid-High** | **High** |
| **Entropy** | Low ($<0.3$) | Moderate | Low-Mid | **Very Low** | **High ($>0.7$)** |
| **Flatness** | Near 0 | Low | Near 0 | **Near 0** | **High** |
| **Spectral Edge** | Low ($<10\text{ Hz}$) | Moderate | Moderate | **Focused ($<15\text{ Hz}$)** | **High / Wide** |
| **Dominant Freq** | Low ($1\text{--}1.5\text{ Hz}$) | Low ($1\text{--}2\text{ Hz}$) | Mid ($2\text{--}4\text{ Hz}$) | **High ($3\text{--}6\text{ Hz}$)** | **Unstable / High** |
| **Harmonics** | **Strong ($1f, 2f, 3f$)** | Present but messy | Strong ($2:1$ or $4:1$ blocks) | **Very Strong (Sawtooth)** | **Absent / Weak** |
| **HNR** | High ($>10\text{ dB}$) | Moderate | High | **Very High ($>15\text{ dB}$)** | **Low ($<3\text{ dB}$)** |
| **VF Filter Leak** | Low | Moderate | Low | **Low** | **High** |
| **Band Power** | Low Bias | Balanced | Balanced | **High Bias** | **High Bias** |

---

### Heuristic Thresholds for VFL vs. VF

This logic assumes you have first ruled out Normal Sinus Rhythm (NSR) by checking if the Heart Rate is normal ($< 120\text{ BPM}$).

#### 1. The "VFL" Signature (The Sawtooth)

* **Concept:** Fast, highly organized, sinusoidal.
* **Primary Checks:**
* **Rate:** Dominant Frequency $> 3.5\text{ Hz}$ ($210\text{ BPM}$).
* **Organization:** Spectral Entropy $< 0.5$ (Low chaos).
* **Purity:** HNR $> 10\text{ dB}$ (Strong peaks).
* **Flatness:** $< 0.1$ (Very spiky).



#### 2. The "VF" Signature (The Chaos)

* **Concept:** Fast, disorganized, broad-spectrum noise.
* **Primary Checks:**
* **Rate:** Dominant Frequency often $> 4\text{ Hz}$ (but can be unstable).
* **Disorganization:** Spectral Entropy $> 0.65$ (High chaos).
* **Noise:** HNR HNR $< 4\text{ dB}$ (Peaks are dissolving).
* **Leak:** VF Filter Leak $> 0.7$ (Most energy is NOT in peaks).



#### 3. Proposed Decision Tree (Heuristic)

```python
def classify_rhythm(metrics):
    # metrics is an object with the calculated values
    
    # 1. Check for SLOW rhythms (NSR, Bradycardia)
    if metrics.dominant_freq < 2.5: # Less than 150 BPM
        return "NSR / OTHER"

    # 2. Check for VENTRICULAR FLUTTER (Fast & Organized)
    # VFL looks like a pure sine wave -> Low Entropy, High HNR
    if (metrics.dominant_freq > 3.5 and 
        metrics.spectral_entropy < 0.5 and 
        metrics.hnr > 9.0):
        return "VFL (Extreme Danger)"

    # 3. Check for VENTRICULAR FIBRILLATION (Fast & Chaotic)
    # VF looks like colored noise -> High Entropy, Low HNR, High Leak
    if (metrics.spectral_entropy > 0.65 and 
        metrics.hnr < 5.0 and
        metrics.vf_filter_leak > 0.6):
        return "VF (Shockable)"
    
    # 4. Grey Zone (Fast VT or noisy AFIB)
    return "VT / NOISY TACHYCARDIA"

```

In [ ]:
# IMPORTS ###########################################

import importlib
import warnings
warnings.filterwarnings("ignore")

class StopExecution(Exception):
    def _render_traceback_(self): pass
pass #class

def Stop():
    print(">>> STOP")
    raise StopExecution()
pass #def

#####################################################

import math
import numpy as np
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt

from pxg import plot
from pxg import EXG, Rids, Record, MS, MV

importlib.reload(plot)

from IPython.display import Markdown

%config InlineBackend.figure_format = "retina"

In [ ]:
## FILTERS ######################

import pywt

def SymletFilter(sig: np.ndarray, wavelet='sym4', level=5, mode='periodic') -> tuple[np.ndarray, np.ndarray]:
    """
    Apply Symlet wavelet decomposition to separate low and high frequency components.
    
    Parameters:
    -----------
    sig : np.ndarray
        Input signal
    wavelet : str
        Symlet wavelet family ('sym4', 'sym5', etc.)
    level : int
        Decomposition level
    mode : str
        Signal extension mode ('periodic', 'zero', 'smooth', etc.)
    
    Returns:
    --------
    low_pass : np.ndarray
        Approximation coefficients (low frequency components)
    high_pass : np.ndarray
        Reconstructed detail coefficients (high frequency components)
    """
    # Perform wavelet decomposition
    coeffs = pywt.wavedec(sig, wavelet, level=level, mode=mode)
    
    # coeffs[0] = approximation coefficients (cA)
    # coeffs[1:] = detail coefficients (cD1, cD2, ..., cDn)
    
    # Reconstruct low-pass signal (approximation only)
    coeffs_low = [coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]]
    low_pass = pywt.waverec(coeffs_low, wavelet, mode=mode)
    
    # Reconstruct high-pass signal (details only)
    coeffs_high = [np.zeros_like(coeffs[0])] + list(coeffs[1:])
    high_pass = pywt.waverec(coeffs_high, wavelet, mode=mode)
    
    # Adjust length to match input (waverec may produce slightly different length)
    low_pass = low_pass[:len(sig)]
    high_pass = high_pass[:len(sig)]
    
    return low_pass, high_pass
pass #def

def SymletBandpass(sig: np.ndarray, wavelet='sym4', level=5, keep_levels=None, mode='periodic') -> np.ndarray:
    """
    Apply Symlet wavelet bandpass filter by keeping specific detail levels.
    
    Parameters:
    -----------
    sig : np.ndarray
        Input signal
    wavelet : str
        Symlet wavelet family
    level : int
        Decomposition level
    keep_levels : list[int] or None
        Which detail levels to keep (1 = highest freq, level = lowest freq)
        If None, keeps all detail levels
    mode : str
        Signal extension mode
    
    Returns:
    --------
    filtered : np.ndarray
        Bandpass filtered signal
    """
    # Perform wavelet decomposition
    coeffs = pywt.wavedec(sig, wavelet, level=level, mode=mode)
    
    if keep_levels is None:
        keep_levels = list(range(1, level + 1))
    pass #if
    
    # Zero out unwanted coefficients
    coeffs_filtered = [np.zeros_like(coeffs[0])]  # Zero approximation
    for i in range(1, len(coeffs)):
        if i in keep_levels:
            coeffs_filtered.append(coeffs[i])
        else:
            coeffs_filtered.append(np.zeros_like(coeffs[i]))
        pass #if
    pass #for
    
    # Reconstruct signal
    filtered = pywt.waverec(coeffs_filtered, wavelet, mode=mode)
    filtered = filtered[:len(sig)]
    
    return filtered
pass #def

def jakova_filter(y, W: int):
    cf = 0.3 * 250 / (W/2)
    print(cf)

    fc = 1.0
    fs = 250.0

    # First order high-pass filter
    # Calculate alpha
    alpha = 1 / (1 + (2 * np.pi * fc / fs)) # approx 0.975
    b = [alpha, -alpha]
    a = [1, -alpha]
    y = signal.lfilter(b, a, y) # type: ignore
    y = signal.lfilter(b, a, y) # type: ignore

    # Lynn filter
    a = np.array([1, -2, 1])
    b = np.zeros(W + 1)
    b[0] = 1
    b[W//2] = -2
    b[W] = 1
    g = (len(b) // 2) ** 2

    y = signal.lfilter(b, a, y) / g # type: ignore
    shift = W // 2 - 1
    y = np.roll(y, -shift)

    # The Jekova Equation
    # y[i] = (14*y[i-1] - 7*y[i-2] + (x[i] - x[i-2])/2) / 8

    b = [1/16, 0, -1/16]
    a = [1, -28/16, 14/16]
    y = signal.lfilter(b, a, y) * 8 # type: ignore
    y = np.roll(y, -1)
    
    return y
pass #def

In [ ]:
from scipy import ndimage

LYN_WIND = 40 // MS   # 40 ms window for the Lynn filter, aprox 15 Hz cutoff frequency
LYN_HIGH = 100 // MS  # 0 ms for the high pass filter (not used)
MED_WIND = 596 // MS  # 596 ms window for the median filter

def LynnFilter(sig: np.ndarray, W: int) -> np.ndarray:
    W -= W % 2
    # cf = 0.3 * FS / (W/2)
    # print(cf)

    a = np.array([1, -2, 1])
    b = np.zeros(W + 1)
    b[0] = 1
    b[W//2] = -2
    b[W] = 1
    g = (len(b) // 2) ** 2
    shift = W // 2 - 1

    sig = signal.lfilter(b, a, sig) / g # type: ignore
    sig = np.roll(sig, -shift)    
    return sig
pass #def

def BaselFilter(sig: np.ndarray, M: int) -> np.ndarray:
    if M <= 1: return sig
    # calculate moving median
    med = ndimage.median_filter(sig, size=M, mode="nearest")
    # med = np.convolve(med, np.ones(M)/M, mode="same")
    sig = sig - LynnFilter(med, W = M)
    return sig
pass #def

def SignalFilter(sig: np.ndarray, L = LYN_WIND, M = MED_WIND * 1, H = LYN_HIGH * 0) -> np.ndarray:
    sig = BaselFilter(sig, M)
    sig = LynnFilter(sig, W = L)
    if H > 0:
        hi = np.convolve(sig, np.ones(H)/H, mode="same")
        sig = sig - hi
    pass #if
    return sig
pass #def

In [ ]:
## PLOT #########################

def PlotPage(
    rec: Record, 
    page = 0, 
    offset = 0,
    marker = False,
    include = []
):
    res = plot.Page(
        rec, 
        page, 
        offset,

        LOW = -700,
        ZEROS = [0, -400, -600],

        SENSOR = False,
        SIGNAL = True,
        DECTOR = False,
        EXTEND = None,

        label = "",
        angle = 0,

        rrqs  = False, 
        qsvl  = False, 
        
        punts = False,  
        trig  = False,
        onoff = False,
        letra = False,

        grid  = False,
        anref = True,
        simple = True,

        # simple = False,
        marker = marker and page not in include,

        show = False,
    )

    if res == []: 
        plot.Show()
        return res
    pass #if

    PON, POF = plot.Range(rec, page, offset)

    plot.Signal(rec.Local[PON:POF] / MV, zero=-400, color="tab:blue", format="-", linewidth=0.6)
    plot.Signal(rec.Level[PON:POF] / MV, zero=-400, color="tab:green", format="-", linewidth=0.5)
    plot.Signal(rec.Peaks[PON:POF] / MV, zero=-400, color="tab:red", format="-")
    
    # plot.Signal(rec.Minim[PON:POF] / MV, zero=-600, color="tab:orange", format="-", linewidth=0.5)
    # plot.Signal(rec.Maxim[PON:POF] / MV, zero=-600, color="tab:green", format="-", linewidth=0.5)

    # Signal(rec.Digit[PON:POF] / MV, zero=-600, color="tab:red", format="-", linewidth=1)

    plot.Show()
    
    return res
pass #def

def PlotRecord(rec: Record, page: int | list[int] = -1, off = 0, marker = False, include = []):
    offset = off*plot.FS
    display(Markdown(f"### {rec.DB.upper()} {rec.RID}"))
    if isinstance(page, list):
        for p in page:
            res = PlotPage(rec, p, offset, marker = marker, include = include)
            # Report(res)
        pass #for
    elif page != -1:
        res = PlotPage(rec, page, offset, marker = marker, include = include)
        # Report(res)
    else:
        for page in range(0, len(rec.Signal) // plot.CHUNK, 1):
            res = PlotPage(rec, page, offset, marker = marker, include = include)
            # Report(res)
        pass #for
    pass #if
    return rec
pass #def

In [ ]:
MIN_PEAK = 4
RAD_PEAK = 40 // MS
DIS_PEAK = 160 // MS

LYN_WIND = 40 // MS
MED_WIND = 596 // MS

AVG_FACTOR = 4
THR_FACTOR = 4

def Detect(rec: Record) -> np.ndarray:
    sig = rec.Sensor

    sig = SignalFilter(sig, L = LYN_WIND, M = MED_WIND)
    # sig = ButterFilter(sig, low=2.0, high=12.0, order=4)
    # sig = FirFilter(sig, low=2.0, high=12.0, order=101)
    
    # sig = SymletBandpass(sig, wavelet='sym4', level=6, keep_levels= [4, 5, 6]) * 4

    rec.Local = sig

    minim = np.sqrt(np.convolve(sig ** 2, np.ones(125)/125, mode="same"))
    maxim = np.sqrt(np.convolve(sig ** 2, np.ones(31)/31, mode="same"))

    # rec.Maxim = np.where(maxim - minim > 0, 100, 0)
    
    # rec.Maxim = maxim
    # rec.Minim = minim

    # rec.Maxim = np.where(maxim > minim, 100, 0)

    pic = sig * 0
    lev = sig * 0

    amp = 0
    thr = 0

    pos_ix = 0
    pos_val = 0

    neg_ix = 0
    neg_val = 0

    for ix in range(RAD_PEAK, len(sig) - RAD_PEAK):
        lev[ix] = thr

        val = sig[ix]
        a = sig[ix - RAD_PEAK] - val
        b = sig[ix + RAD_PEAK] - val
        if a * b <= 0: continue
        if min(abs(a), abs(b)) < MIN_PEAK: continue

        found = False
        if val > 0:
            if ix - pos_ix < DIS_PEAK and abs(val) >= abs(pos_val):
                pic[pos_ix] = 0
                found = True
            elif ix - pos_ix >= DIS_PEAK and abs(val) >= thr:
                found = True
            pass #if
            if found:
                pic[ix] = val
                amp += (abs(val) - amp) / AVG_FACTOR
                thr = amp / THR_FACTOR
                pos_ix = ix
                pos_val = val
            pass #if
        else:
            if ix - neg_ix < DIS_PEAK and abs(val) >= abs(neg_val):
                pic[neg_ix] = 0
                found = True
            elif ix - neg_ix >= DIS_PEAK and abs(val) >= thr:
                found = True
            pass #if
            if found:
                pic[ix] = val
                amp += (abs(val) - amp) / AVG_FACTOR
                thr = amp / THR_FACTOR
                neg_ix = ix
                neg_val = val
            pass #if
        pass #if
    pass #for

    rec.Peaks = pic
    rec.Level = lev
    p = np.sign(pic)

    rec.Digit = p * 100
    return pic

    bx = 0
    bv = 0
    cx = 0
    cv = 0

    for ix in range(1, len(p) - 1):
        val = p[ix]
        if val == 0: continue

        bx = cx
        bv = cv
        cx = ix
        cv = val

        if np.sign(bv) == np.sign(cv):
            p[bx] = 0
        pass #if
    pass #for

    p = np.convolve(np.abs(p), np.ones(500), mode="same")
    rec.Digit = np.where(p > 7, 1, 0) * 200

    return pic
pass #def


In [ ]:
def PlotVF(db: str, rid: str):
    rec = Record(db, rid)

    Detect(rec)
    
    for b in rec.Lines:
        if b.Episode in ["VFL", "VT", "VF", "#AFL"]:
            b.Marker = b.Episode
        pass #if
    pass #for
    PlotRecord(rec, page=-1, off=0, marker=True, include= [0])
pass #def

def PlotDB(db: str = "mitdb", recs = []):   
    if len(recs) > 0:
        rids = recs
    else:
        rids = Rids(db) # Run(db, learn=True, cfm=True, dev=False)
    pass #if
    for rid in rids:
        PlotVF(db, rid)
        # rec = Record(DBMIT, rid)
        # PlotRecord(rec, page=0, off=0)
    pass #for
pass #def


# Run("mitdb", learn=True, cfm=True, wrx = False, dev=False)
# Run("vfdb", learn=True, cfm=True, wrx = False, dev=False)
# Run("cudb", learn=True, cfm=True, wrx = False, dev=False)
# Run("ahadb", learn=True, cfm=True, wrx = False, dev=False)

PlotDB("mitdb", ["100", "102", "105", "118", "119", "207", "222"])
# PlotDB("vfdb", [])